# Lab 3: N-Grams - Generating tweets using n-grams

In [ ]:
import re
import glob
import random
from itertools import chain

import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer
from nltk.util import ngrams
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

try:
    import emoji
except ImportError:
    %pip install -q emoji
    import emoji

## 1. Load the data

In [ ]:
csv_file = glob.glob('*.csv')[0]
df = pd.read_csv(csv_file)

# the column that holds the tweet text = the text column with the longest average length
text_col = df.select_dtypes(exclude='number').apply(lambda s: s.astype(str).str.len().mean()).idxmax()
tweets = df[text_col].dropna().astype(str).tolist()

print('File   :', csv_file)
print('Column :', text_col)
print('Tweets :', len(tweets))
df.head()

## 2. Pre-processing

In [ ]:
def clean_tweet(text):
    text = re.sub(r'\bRT\b', ' ', text)                    # remove RT
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)      # remove websites
    text = re.sub(r'@\w+:?', ' ', text)                       # remove mentions
    text = re.sub(r'#\w+', ' ', text)                        # remove hashtags
    text = emoji.replace_emoji(text, replace=' ')            # remove emojis
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

clean_tweets = [clean_tweet(t) for t in tweets]
clean_tweets = [t for t in clean_tweets if t]          # drop tweets that became empty

for before, after in list(zip(tweets, clean_tweets))[:5]:
    print('BEFORE:', ' '.join(before.split()))
    print('AFTER :', after)
    print()
print('Tweets after cleaning:', len(clean_tweets))

## 3. Build the MLE bigram model

In [ ]:
tokenized = [word_tokenize(t) for t in clean_tweets]

# train / test split (90% / 10%)
random.Random(42).shuffle(tokenized)
split = int(0.9 * len(tokenized))
train_tokens, test_tokens = tokenized[:split], tokenized[split:]

train, vocab = padded_everygram_pipeline(2, train_tokens)

lm = MLE(2)
lm.fit(train, vocab)

print('Vocabulary size:', len(lm.vocab))
print('Train tweets   :', len(train_tokens))
print('Test tweets    :', len(test_tokens))

### Generate tweets

In [ ]:
detok = TreebankWordDetokenizer()

def generate_tweet(max_words=30, seed=0):
    words = ['<s>']
    for i in range(max_words):
        word = lm.generate(1, text_seed=words[-1:], random_seed=seed + i)
        if word == '</s>':
            break
        words.append(word)
    return detok.detokenize(words[1:])

for k in range(5):
    print(f'{k + 1}. {generate_tweet(seed=k * 100)}')

## 4. Evaluate the model

In [ ]:
test_bigrams = list(chain.from_iterable(ngrams(pad_both_ends(t, n=2), 2) for t in test_tokens))

# bigrams of the test set that also appear in the training data
seen = [b for b in test_bigrams if lm.score(b[1], [b[0]]) > 0]

print('Test bigrams                 :', len(test_bigrams))
print('Seen in training             :', len(seen), f'({len(seen) / len(test_bigrams):.1%})')
print('Perplexity (all test bigrams):', lm.perplexity(test_bigrams))   # inf if any bigram is unseen (MLE gives it 0)
print('Perplexity (seen bigrams)    :', lm.perplexity(seen))

## 5. Probability of the bigram (pakistan is)

In [ ]:
print("P(is | pakistan) =", lm.score('is', ['pakistan']))

## 6. Perplexity of the word (pakistan)

In [ ]:
print("Perplexity of 'pakistan' =", lm.perplexity([('pakistan',)]))